In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

x_direction_range = np.linspace(-np.deg2rad(5), np.deg2rad(5), 9)
z_direction_range = np.linspace(-np.deg2rad(5), np.deg2rad(5), 9)
orientation_range = np.arange(-90, 91, 5)
depth_range = [0,15, 0.16, 0.17, 0.18, 0.19, 0.20, 0.21, 0.22, 0.23, 0.24, 0.25]
total_count = len(x_direction_range)*len(z_direction_range)*len(depth_range)*len(orientation_range)*12
print(total_count, total_count/12)

basis_log_fname = f'data/basis_log.csv'
basis_log = pd.read_csv(basis_log_fname)
basis_pupill_data = basis_log[["center_x","center_y","confidence"]]
basis_pupill_data = basis_pupill_data.to_numpy()

basis = []
basis_params = []
idx = 0
for x, x_d in enumerate(x_direction_range):
    for z, z_d in enumerate(z_direction_range):
        for depth in depth_range:
            for d, deg in enumerate(orientation_range):
                tmp_basis = np.zeros((12,2))
                for t in range(12):
                    tmp_basis[t,:] = basis_pupill_data[idx, :2]
                    idx += 1
                if np.any(tmp_basis == -1):
                    pass
                else:
                    basis.append(tmp_basis)
                    basis_params.append([x_d, z_d, depth, deg])

basis = np.array(basis)
np.save("data/basis.npy", basis)

basis_params = np.array(basis_params)
np.save("data/basis_params.npy", basis_params)

431568 35964.0


In [3]:
import pickle
import mocet
import os
from multiprocessing import Pool
from multiprocessing import get_context
import time

testable_data = pickle.load(open('../testable_data_list.pkl', 'rb'))

subject_pool = {
    'sub-003': {'ses-07R': ([1, 2, 3, 4, 5], False), 'ses-13R': ([1, 2, 4, 5, 6], False)},
    'sub-004': {'ses-07R': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-005': {'ses-07': ([1, 2, 3, 4, 5, 6], True)},
    'sub-006': {'ses-07R': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-008': {'ses-07R': ([2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-009': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 5, 6], False)},
    'sub-010': {'ses-07': ([1, 2, 3, 4, 5], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-011': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-012': {'ses-07': ([1, 2, 4, 5, 6], False)},
    'sub-013': {'ses-07': ([1, 2, 3, 4], False)},
    'sub-014': {'ses-07': ([2, 3, 4, 5, 6], False)},
    'sub-015': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-016': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-017': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5], False)},
    'sub-018': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-020': {'ses-07': ([1, 2, 3, 4, 5, 6], False), 'ses-13': ([1, 2, 3, 4, 5, 6], False)},
    'sub-021': {'ses-07': ([1, 2, 4, 5, 6], False), 'ses-13': ([1, 2, 4, 5, 6], False)},
    'sub-JJY': {'ses-07': ([1, 2, 3, 4, 5, 6], False)},
    'sub-KMY': {'ses-07': ([1, 2, 3, 4, 5, 6], False)},
    'sub-PJW': {'ses-07': ([1, 2, 3, 4, 6], True)},
    'sub-PBJ': {'ses-07': ([1, 2, 3, 4, 5], False)}
}

history_onset = {'sub-005': [28.66, 29.32, 28.12, 33.7, 36.1, 27.46], 'sub-PJW': [35, 30.8, 28.66, 26.58, None, 27.42]}

task_duration = 816
calibration_onsets = [1, 494]
calibration_points = [24, 12]
interval = 1.6
calibration_offset_start = 0.5
calibration_offset_end = -0.5
t_cal = 0
n_iterations = 100 # iterations for each subject

motion_param_labels = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
dir_path = f'/Users/jiwoongpark/Eyetracking_data/null_simulation'
if not os.path.exists(dir_path):
    os.mkdir(dir_path) # create directory

def random_phase_shuffle(data):
    fft_data = np.fft.fft(data, axis=0)
    amplitude = np.abs(fft_data)
    phase = np.angle(fft_data)
    random_phase = np.random.uniform(0, 2 * np.pi, phase.shape)
    new_fft_data = amplitude * np.exp(1j * random_phase)
    return np.real(np.fft.ifft(new_fft_data, axis=0))

def generate_null_simulation(index):
    np.random.seed(index) # for reproducibility
    random_motion_params = random_phase_shuffle(motion_params)
    random_motion_params = random_motion_params - random_motion_params[0,:]
    mocet.simulation.generate(random_motion_params, basis_params[basis_idx], dir_path, index=index, render=True)

index_baseline = 0
time_sta = time.time()
for key in testable_data.keys():
    subject = key[0]
    session = key[1]
    task = key[2]
    run = key[3]
    r = int(run[4])

    root = f'../../_DATA/{subject}/{session}'
    log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
    data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt'
    confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'

    runs, history_loss = subject_pool[subject][session]
    if history_loss: 
        start = history_onset[subject][r-1]
    else:
        history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
        start, _, _ = mocet.utils.get_avotec_history(history_fname)

    pupil_data, pupil_timestamps, pupil_confidence, pupil_diameter = mocet.utils.clean_avotec_data(log_fname,
                                                                                            data_fname,
                                                                                            start=start,
                                                                                            duration=task_duration)
    offset = calibration_onsets[t_cal]
    calibration_pupils = []
    for i in np.arange(calibration_points[t_cal]):
        start = (offset+i)*interval + calibration_offset_start
        end = (offset+i+1)*interval + calibration_offset_end
        log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
        calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                                  np.nanmean(pupil_data[log_effective,1])])
    calibration_pupils = np.array(calibration_pupils).reshape(2,12,2).mean(axis=0)
    
    basis_similarity = []
    for i in range(len(basis)):
        basis_similarity.append(np.mean(np.sqrt(np.sum((basis[i] - calibration_pupils)**2, axis=-1))))
    basis_idx = np.argmin(basis_similarity)
    print(subject, session, task, run, r, basis_idx)
    
    fmriprep_confounds = pd.read_csv(confounds_fname, delimiter='\t')
    motion_params = fmriprep_confounds[motion_param_labels]
    motion_params = np.nan_to_num(motion_params)
    
    p = get_context("fork").Pool(8)
    results = p.map(generate_null_simulation, [index_baseline + (510*i) for i in range(n_iterations)])
    p.close()
    index_baseline += 510 * n_iterations
    
    print(f"Elapsed: {time.time() - time_sta:3.3f}")
    time_sta = time.time()

sub-003 ses-07R task-mcHERDING run-1 1 10977
Elapsed: 1252.645
sub-003 ses-07R task-mcHERDING run-2 2 10792
Elapsed: 1312.653
sub-003 ses-07R task-mcHERDING run-3 3 7188
Elapsed: 1308.411
sub-003 ses-13R task-mcHERDING run-1 1 17187
Elapsed: 1261.401
sub-003 ses-13R task-mcHERDING run-2 2 16785
Elapsed: 1264.309
sub-003 ses-13R task-mcHERDING run-4 4 16743
Elapsed: 1274.184
sub-003 ses-13R task-mcHERDING run-5 5 16750
Elapsed: 1284.172
sub-004 ses-07R task-mcHERDING run-1 1 10491
Elapsed: 1293.665
sub-004 ses-07R task-mcHERDING run-2 2 10825
Elapsed: 1276.909
sub-004 ses-07R task-mcHERDING run-4 4 11305
Elapsed: 1249.629
sub-004 ses-13 task-mcHERDING run-1 1 24080
Elapsed: 1273.698
sub-004 ses-13 task-mcHERDING run-3 3 18334
Elapsed: 1241.144
sub-004 ses-13 task-mcHERDING run-6 6 24561
Elapsed: 1237.413
sub-005 ses-07 task-mcHERDING run-1 1 10493
Elapsed: 1281.455
sub-005 ses-07 task-mcHERDING run-3 3 10526
Elapsed: 1286.620
sub-006 ses-07R task-mcHERDING run-1 1 10564
Elapsed: 1262.73

In [3]:
import numpy as np

age = [28.53, 25.55, 24.59, 23.3, 21.98, 
       19.88, 22.25, 22.01, 19.94, 19.41, 26.04, 31.07, 
       21.98, 22.61, 23.95, 26.1, 29.1, 23.5, 28.4, 23.5]
print(np.mean(age), np.std(age))

24.1845 3.155560924780252
